<a href="https://colab.research.google.com/github/Ravi-ranjan1801/ML-Lab/blob/main/ml_lab_05_anomaly_detection_for_human_drivers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install eclipse-sumo tensorflow scikit-learn pandas numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.2/140.2 MB 6.7 MB/s eta 0:00:00


In [ ]:
import os, sys, subprocess, textwrap, random
from collections import defaultdict, deque

import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

import sumo

os.environ["SUMO_HOME"] = sumo.SUMO_HOME
sys.path.append(os.path.join(os.environ["SUMO_HOME"], "tools"))

import traci

SUMO_BIN = os.path.join(os.environ["SUMO_HOME"], "bin", "sumo")
NETCONVERT_BIN = os.path.join(os.environ["SUMO_HOME"], "bin", "netconvert")

WORKDIR = "/content/driver_anomaly"
os.makedirs(WORKDIR, exist_ok=True)

print("SUMO_HOME =", os.environ["SUMO_HOME"])
print("SUMO_BIN =", SUMO_BIN)
print("NETCONVERT_BIN =", NETCONVERT_BIN)

SUMO_HOME = /usr/local/lib/python3.12/dist-packages/sumo
SUMO_BIN = /usr/local/lib/python3.12/dist-packages/sumo/bin/sumo
NETCONVERT_BIN = /usr/local/lib/python3.12/dist-packages/sumo/bin/netconvert


In [ ]:
nodes_xml = """<nodes>
    <node id="n0" x="0.0" y="0.0" type="priority"/>
    <node id="n1" x="1000.0" y="0.0" type="priority"/>
</nodes>"""

edges_xml = """<edges>
    <edge id="e0" from="n0" to="n1" numLanes="3" speed="27.78"/>
</edges>"""

with open(f"{WORKDIR}/road.nod.xml", "w") as f:
    f.write(nodes_xml)

with open(f"{WORKDIR}/road.edg.xml", "w") as f:
    f.write(edges_xml)

subprocess.run([
    NETCONVERT_BIN,
    "--node-files", f"{WORKDIR}/road.nod.xml",
    "--edge-files", f"{WORKDIR}/road.edg.xml",
    "--output-file", f"{WORKDIR}/road.net.xml"
], check=True)

vehicles = []
depart = 0.0
for i in range(120):
    vehicles.append(
        f'<vehicle id="veh{i}" type="car" route="r0" depart="{depart:.1f}" departLane="free" departSpeed="max"/>'
    )
    depart += 0.8

routes_xml = f"""<routes>
    <vType id="car"
           accel="2.6"
           decel="4.5"
           sigma="0.2"
           tau="1.0"
           length="5.0"
           minGap="2.5"
           maxSpeed="27.78"
           carFollowModel="IDM" />
    <route id="r0" edges="e0"/>
    {"".join(vehicles)}
</routes>"""

with open(f"{WORKDIR}/road.rou.xml", "w") as f:
    f.write(routes_xml)

sumocfg = """<configuration>
    <input>
        <net-file value="road.net.xml"/>
        <route-files value="road.rou.xml"/>
    </input>
    <time>
        <begin value="0"/>
        <end value="250"/>
        <step-length value="0.1"/>
    </time>
</configuration>"""

with open(f"{WORKDIR}/run.sumocfg", "w") as f:
    f.write(sumocfg)

print("Files created in", WORKDIR)

Files created in /content/driver_anomaly


In [ ]:
def collect_normal_data():
    rows = []
    prev_acc = {}

    traci.start([
        SUMO_BIN,
        "-c", f"{WORKDIR}/run.sumocfg",
        "--no-step-log", "true"
    ])

    while traci.simulation.getMinExpectedNumber() > 0:
        traci.simulationStep()
        t = traci.simulation.getTime()

        for vid in traci.vehicle.getIDList():
            speed = traci.vehicle.getSpeed(vid)
            accel = traci.vehicle.getAcceleration(vid)
            lane_idx = traci.vehicle.getLaneIndex(vid)

            old_acc = prev_acc.get(vid, accel)
            jerk = (accel - old_acc) / 0.1
            prev_acc[vid] = accel

            rows.append([t, vid, speed, accel, jerk, lane_idx])

    traci.close()
    df = pd.DataFrame(rows, columns=["time", "vehicle_id", "speed", "accel", "jerk", "lane_idx"])
    return df

normal_df = collect_normal_data()
normal_df.to_csv(f"{WORKDIR}/normal_traces.csv", index=False)

print(normal_df.head())
print("Rows:", len(normal_df))

 Retrying in 1 seconds
   time vehicle_id  speed         accel          jerk  lane_idx
0   0.1       veh0  27.78  0.000000e+00  0.000000e+00         0
1   0.2       veh0  27.78 -2.006502e-09 -2.006502e-08         0
2   0.3       veh0  27.78 -1.931362e-09  7.513989e-10         0
3   0.4       veh0  27.78 -1.859064e-09  7.229772e-10         0
4   0.5       veh0  27.78 -1.789466e-09  6.959766e-10         0
Rows: 49556


In [ ]:
FEATURES = ["speed", "accel", "jerk", "lane_idx"]
WINDOW = 100
STRIDE = 10   # 1 second shift

scaler = MinMaxScaler()
normal_df[FEATURES] = scaler.fit_transform(normal_df[FEATURES])

def make_windows(df, features, window=100, stride=10):
    X = []
    for vid, g in df.groupby("vehicle_id"):
        g = g.sort_values("time")
        arr = g[features].values
        for start in range(0, len(arr) - window + 1, stride):
            X.append(arr[start:start+window])
    return np.array(X, dtype=np.float32)

X = make_windows(normal_df, FEATURES, WINDOW, STRIDE)
print("X shape =", X.shape)

X_train, X_val = train_test_split(X, test_size=0.2, random_state=42)

X shape = (3822, 100, 4)


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, Model

timesteps = X_train.shape[1]
n_features = X_train.shape[2]

inp = layers.Input(shape=(timesteps, n_features))

x = layers.LSTM(64, return_sequences=True)(inp)
x = layers.LSTM(32, return_sequences=False)(x)

x = layers.RepeatVector(timesteps)(x)

x = layers.LSTM(32, return_sequences=True)(x)
x = layers.LSTM(64, return_sequences=True)(x)

out = layers.TimeDistributed(layers.Dense(n_features))(x)

model = Model(inp, out)
model.compile(optimizer="adam", loss="mse")
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 100, 4)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 100, 64)        │        17,664 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ repeat_vector (RepeatVector)    │ (None, 100, 32)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 100, 32)        │         8,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ (None, 100, 64)        │        24,832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed                │ (None, 100, 4)         │           260 │
│ (TimeDistributed)               │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 63,492 (248.02 KB)

 Trainable params: 63,492 (248.02 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
train_pred = model.predict(X_train, verbose=0)
train_err = np.mean((X_train - train_pred) ** 2, axis=(1, 2))

val_pred = model.predict(X_val, verbose=0)
val_err = np.mean((X_val - val_pred) ** 2, axis=(1, 2))

threshold = np.percentile(train_err, 95)

print("Train error mean:", train_err.mean())
print("Val error mean:", val_err.mean())
print("Threshold:", threshold)

Train error mean: 0.35003904
Val error mean: 0.3522685
Threshold: 0.53161407


In [ ]:
def run_live_detection(threshold):
    detections = []
    buffers = defaultdict(lambda: deque(maxlen=WINDOW))
    prev_acc = {}
    bad_vid = None

    # close leftover TraCI connection from previous failed run
    try:
        if traci.isLoaded():
            traci.close(False)
    except Exception:
        pass

    traci.start([
        SUMO_BIN,
        "-c", f"{WORKDIR}/run.sumocfg",
        "--no-step-log", "true"
    ])

    try:
        while traci.simulation.getMinExpectedNumber() > 0:
            traci.simulationStep()
            t = traci.simulation.getTime()
            current_ids = traci.vehicle.getIDList()

            # choose one vehicle after simulation has started
            if bad_vid is None and t > 30 and len(current_ids) > 0:
                bad_vid = current_ids[0]
                print("Injected anomaly vehicle:", bad_vid)

            # force erratic behavior
            if bad_vid is not None and bad_vid in current_ids and t > 35:
                try:
                    traci.vehicle.setSpeedMode(bad_vid, 96)
                    if int(t) % 4 == 0:
                        traci.vehicle.setSpeed(bad_vid, 28.0)
                    elif int(t) % 4 == 2:
                        traci.vehicle.setSpeed(bad_vid, 4.0)

                    if int(t) % 5 == 0:
                        current_lane = traci.vehicle.getLaneIndex(bad_vid)
                        target_lane = 2 if current_lane < 2 else 0
                        traci.vehicle.changeLane(bad_vid, target_lane, 2.0)
                except Exception as e:
                    print("Anomaly injection error:", e)

            for vid in current_ids:
                speed = traci.vehicle.getSpeed(vid)
                accel = traci.vehicle.getAcceleration(vid)
                lane_idx = traci.vehicle.getLaneIndex(vid)

                old_acc = prev_acc.get(vid, accel)
                jerk = (accel - old_acc) / 0.1
                prev_acc[vid] = accel

                feat_df = pd.DataFrame(
                    [[speed, accel, jerk, lane_idx]],
                    columns=FEATURES
                )
                feat = scaler.transform(feat_df)[0].astype(np.float32)
                buffers[vid].append(feat)

                # score once per second when buffer is full
                if len(buffers[vid]) == WINDOW and int(t * 10) % 10 == 0:
                    window = np.array(buffers[vid], dtype=np.float32)[None, :, :]
                    recon = model.predict(window, verbose=0)
                    err = np.mean((window - recon) ** 2)

                    detections.append({
                        "time": t,
                        "vehicle_id": vid,
                        "error": float(err),
                        "is_anomaly": bool(err > threshold)
                    })

    finally:
        try:
            traci.close(False)
        except Exception:
            pass

    return pd.DataFrame(detections)

detect_df = run_live_detection(threshold)
detect_df.head()

 Retrying in 1 seconds
Injected anomaly vehicle: veh0


,time,vehicle_id,error,is_anomaly
0,10.0,veh0,0.394357,False
1,11.0,veh0,0.394357,False
2,11.0,veh1,0.381812,False
3,12.0,veh0,0.394357,False
4,12.0,veh1,0.381812,False


In [ ]:
flags = detect_df[detect_df["is_anomaly"]].sort_values("error", ascending=False)
print(flags.head(20))

     time vehicle_id     error  is_anomaly
5    12.0       veh2  0.633982        True
8    13.0       veh2  0.633982        True
12   14.0       veh2  0.633982        True
18   15.0       veh2  0.633982        True
25   16.0       veh2  0.633982        True
33   17.0       veh2  0.633982        True
43   18.0       veh2  0.633982        True
55   19.0       veh2  0.633982        True
68   20.0       veh2  0.633982        True
82   21.0       veh2  0.633982        True
98   22.0       veh2  0.633982        True
115  23.0       veh2  0.633982        True
133  24.0       veh2  0.633982        True
152  25.0       veh2  0.633982        True
172  26.0       veh2  0.633982        True
193  27.0       veh2  0.633982        True
215  28.0       veh2  0.633982        True
238  29.0       veh2  0.633982        True
288  31.0       veh2  0.633982        True
262  30.0       veh2  0.633982        True
